# KLTN attack v4-h1 — Counterfactual concepts, CLIP relevance and dynamic regions

**Deliverable:** InternVL3-1B sinh source/target captions như implementation MPC; chỉ **CLIP ViT-B/16 của FOA** (B32/Laion tùy chọn) tham gia gradient attack. Captioner được giải phóng trước khi nạp CLIP. Mỗi variant xuất manifest + ảnh lossless dùng được với `text_gen_eval_v2.ipynb`.

`h1` là tên họ thí nghiệm. Bản mặc định `h1_full` triển khai cả bốn ý, với counterfactual hard negatives là biến chính; các control tách H1/H2/H3/H4. Đây là **proposed hybrid**, không phải reproduction nguyên bản FOA/MPC và chưa có bằng chứng tăng ASR hoặc novelty đã được công nhận.

**Chạy:** Add Input dataset của bạn, bật Internet/GPU; sửa Cell 1. Mặc định `RUN_ATTACK=False`, `NUM_SAMPLES=1` để preflight. Bật `RUN_ATTACK=True` và Run All để chạy thử; sau đó chọn `NUM_SAMPLES=100` hoặc `None`. `RUN_VARIANTS` quyết định ablations. Mặc định dùng một CLIP B16; Laion ViT-G rất lớn, chỉ bật ensemble khi đã kiểm tra RAM/VRAM. Với T4×2 có thể đặt `CAPTION_DEVICE='cuda:1'`; một GPU vẫn chạy được vì caption/attack theo hai giai đoạn. Local captions không phải captions của victim để tính ASR.

**Trace:** console ghi sample/role/step/negative/region/loss/ETA; `events.jsonl`, `text_records.jsonl`, `samples/<ID>/steps.jsonl`, `maps/`, `status.json`, `summary.json` lưu trên đĩa. `RESUME_RUN_DIR` tiếp tục ở **ranh giới mẫu**; mẫu đang dở được chạy lại từ đầu với seed riêng. Fingerprint kiểm tra config, code, revisions và hashes ảnh. Không resume optimizer giữa hai step.

## v4.1 — Fix for the run that generated zero adversarial images

**Incident confirmed in supplied ZIP:** sample `00000` had valid source and target captions but InternVL3-1B failed to produce `source_concepts` / `target_edits` after three attempts. `text_records.jsonl` recorded `status=error`; Cell 12 subsequently wrote `missing_valid_caption` and skipped the optimizer. Output `manifest.jsonl` was empty. This is a preparation-stage failure, **not an evaluated ASR of 0**. The ZIP also shows loading a 5.47 GB LAION ViT-G checkpoint, but it does **not** prove an OOM error.

**Fixes**

1. Generate concise JSON with bounded retries; if it fails, build deterministic *caption-derived* negative edits and label `provenance=deterministic_fallback_unverified`. Such captions are **not verified ground-truth falsehoods**; manually inspect `negative_candidates.json` before scientific reporting. `CONCEPT_FAILURE_POLICY='strict'` disables this recovery.
2. Explicitly record when too few candidates pass the original CLIP filter. `NEGATIVE_FILTER_POLICY='ranked_relaxation'` keeps the lowest-violation candidates and writes `negative_filter_relaxed=true` per sample; use `'strict'` to reproduce the original exclusion criterion. A relaxed result must not be directly equated with a strict-baseline result.
3. Check that some samples have usable captions **before loading CLIP**; do not silently produce another zero-sample run. Successful captions from an interrupted sample may be reused when rerunning in a compatible run.
4. Default to B16 for a one-sample smoke test to reduce Kaggle memory pressure. Explicitly restore the matched surrogate backbone configuration for a controlled ablation. The high-cost ViT-G model is **not** necessary for confirming that steps progress.

**Run on Kaggle:** Attach dataset `zintom/blackbox-adversarial-attack-dataset`, enable Internet and a GPU, choose `NUM_SAMPLES=1`, `STEPS=10` for a quick smoke test or `STEPS=300` for the intended experiment, and **Run All**. Watch for `concept/valid` or `concept/fallback_used`, then `attack/ready_samples`, `attack/sample_start`, `attack/progress` and `attack/sample_complete`. Inspect `summary.json`, `samples/00000/steps.jsonl`, `images/00000_adversarial.png`, and ensure `linf_255 <= EPSILON_255`. For official research claims, audit concepts and re-run with the strict settings and the exact baseline backbone list.

**Validation scope:** Notebook JSON/AST and deterministic fallback unit tests are checked locally. Real GPU/InternVL/CLIP inference still needs a Kaggle runtime with model weights and dataset attached; no real attack completion is claimed here.


## 1. Evidence và giới hạn novelty — đọc ngày 2026-10-13

Các diễn giải dưới đây dựa trên abstract và phần phương pháp được chỉ rõ; **suy luận cho thí nghiệm** được tách khỏi kết quả tác giả. Không dùng việc không tìm thấy một paper làm bằng chứng “chưa ai làm”. Venue chỉ ghi khi nguồn chính thức xác nhận; SAGA/GDA/SADCA không được tự gắn NeurIPS/CVPR từ tên file. IEEE S&P, USENIX Security, CCS, NDSS là conference venues, không phải journal.

| Nguồn chính thức, phần đọc | Điều paper hỗ trợ (diễn giải) | Vai trò ở đây |
|---|---|---|
| [FOA, NeurIPS 2025](https://proceedings.nips.cc/paper_files/paper/2025/hash/540a3bf4863672cdaebfed69fa2ce335-Abstract-Conference.html), abstract; [paper §3](https://arxiv.org/abs/2505.21494) | Kết hợp global cosine, local clustering/OT và dynamic ensemble weighting để cải thiện transfer. | Backbone và FOA-derived distance; không claim loss này mới. |
| [MPCAttack, CVPR 2026](https://openaccess.thecvf.com/content/CVPR2026/papers/Li_Multi-Paradigm_Collaborative_Adversarial_Attack_Against_Multi-Modal_Large_Language_Models_CVPR_2026_paper.pdf), [§3.2 Eq.3–6, §4.1](https://arxiv.org/abs/2603.04846); [author code](https://github.com/LiYuanBoJNU/MPCAttack) | Sinh caption, mã hóa text bằng CLIP, trộn ảnh/text; contrastive target attraction/source repulsion. Full method có CLIP, InternVL và DINO. | Giữ captioner InternVL3-1B và joint anchors; loại InternVL/DINO khỏi gradient theo yêu cầu. H4 riêng lẻ đã có overlap trực tiếp. |
| [Gradient-based Visual Explanation for Transformer-based CLIP (Grad-ECLIP), ICML 2024](https://proceedings.mlr.press/v235/zhao24p.html), abstract | Giải thích matching ảnh–text của CLIP bằng channel/spatial weights trên token features; self-attention của CLIP có thể quá sparse. | XAI comparator cần chạy sau; **không gọi rollout trong notebook là Grad-ECLIP**. |
| [Chefer et al., ICCV 2021](https://openaccess.thecvf.com/content/ICCV2021/papers/Chefer_Generic_Attention-Model_Explainability_for_Interpreting_Bi-Modal_and_Encoder-Decoder_Transformers_ICCV_2021_paper.pdf), [abstract, §3](https://arxiv.org/abs/2103.15679) | Relevance propagation cho attention models, gồm mô hình multimodal. | Gradient-weighted attention rollout trong **vision branch** CLIP, có text-conditioned score. |
| [Finer-CAM, CVPR 2025](https://openaccess.thecvf.com/content/CVPR2025/papers/Zhang_Finer-CAM_Spotting_the_Difference_Reveals_Finer_Details_for_Visual_Explanation_CVPR_2025_paper.pdf), abstract | So sánh target với lớp gần để làm nổi bật evidence phân biệt thay vì evidence dùng chung. | Suy luận: dùng concept-margin để conditioning rollout; đây không phải implementation Finer-CAM. |
| [FIA, ICCV 2021](https://openaccess.thecvf.com/content/ICCV2021/papers/Wang_Feature_Importance-Aware_Transferable_Adversarial_Attacks_ICCV_2021_paper.pdf), abstract | Aggregate gradients qua transformed images để tập trung object-aware features khi transfer attack. | Bằng chứng “importance-guided transfer” đã tồn tại; baseline XAI cần đối chiếu. |
| [CF-VLM, NeurIPS 2025](https://papers.nips.cc/paper_files/paper/2025/file/541b6d155146d142a5e10d787d1d430f-Paper-Conference.pdf), [abstract](https://arxiv.org/abs/2506.17267) | Counterfactual samples và ba training objectives cho fine-grained discrimination/causal sensitivity. | H1 có tiền lệ ở training; notebook đóng băng weights và chỉ tối ưu pixels, không claim causal identification. |
| [GDA](https://arxiv.org/abs/2602.09431v2), [§4.1–4.3, Algorithm 1](https://arxiv.org/html/2602.09431v2), nguồn ghi under review | Text-grounded perturbation allocation và global/local evidence disruption, dùng Grad-ECLIP. | H2 đã có overlap rất lớn. Caption/mask ban đầu trong Algorithm 1 không tự chứng minh dynamic rule ở đây mới. |
| [SAGA](https://arxiv.org/abs/2602.04356v2), abstract, nguồn ghi preprint | Fixed surrogate-LVLM attention map dẫn region sequencing qua các stage. | H3 cần so với fixed map và fresh map; “region sequence” riêng lẻ không mới. |
| [V-Attack](https://arxiv.org/abs/2511.20223), [author repo ghi CVPR 2026](https://github.com/Summu77/V-Attack), abstract | Text-guided value manipulation định vị source concept và chuyển về target concept. | Concept-local attack đã có; rollout không giải quyết tự động patch semantic entanglement mà paper nêu. |
| [SADCA](https://arxiv.org/abs/2603.04839v3), [abstract, §3.2 Eq.3–6](https://arxiv.org/html/2603.04839v3), tác giả ghi accepted CVPR 2026 | Positive/negative samples, dynamic image/text interactions và semantic augmentation cho transfer VLP attacks. | H1/H3/H4 có overlap. Ở đây text bank cố định, chỉ ảnh bị perturb; hard selection và relevance cập nhật theo ảnh hiện tại. |

| [PDCL-Attack, ECCV 2024](https://arxiv.org/abs/2407.20657v2), abstract và metadata | Dùng CLIP, prompt-driven text guidance từ ground-truth class labels cho generative transfer attack. | Text-guided contrastive transfer đã có; ở đây tối ưu pixels từng cặp với caption, không học prompt/generator. |
| [dSVA, ICCV 2025](https://openaccess.thecvf.com/content/ICCV2025/html/Wu_Boosting_Generative_Adversarial_Transferability_with_Self-supervised_Vision_Transformer_Features_ICCV_2025_paper.html), abstract | Generator khai thác features và attention của hai self-supervised ViTs để tăng transfer. | Attention-guided transferable attack không mới; không thêm DINO/MAE vào notebook theo yêu cầu. |
| [Explainable Saliency / XSal, CVPR 2025](https://openaccess.thecvf.com/content/CVPR2025/papers/Chen_Explainable_Saliency_Articulating_Reasoning_with_Contextual_Prioritization_CVPR_2025_paper.pdf), abstract | Reasoning bằng VLM và contextual prioritization cho saliency kèm giải thích ngôn ngữ. | Gợi ý ưu tiên concept theo ngữ cảnh; paper không chứng minh tăng adversarial ASR. Không tái tạo XSal trong notebook. |
| [IAU-FOA, arXiv 2610.06977](https://arxiv.org/abs/2610.06977), abstract; preprint tháng 10/2026 | Global/local alignment, confidence-adaptive unbalanced transport và visual-invariance augmentation. | Related work mới để kiểm tra overlap; không coi adaptive transport là đóng góp của notebook. |

Các paper XAI nên đọc để thiết kế thí nghiệm là **Grad-ECLIP → Chefer → Finer-CAM**; FIA/dSVA/PDCL và các attack phía trên là tiền lệ phải đối chiếu, không đề xuất lại như novelty. Preprints được giữ trong kiểm tra overlap dù chưa xác nhận venue trong danh sách yêu cầu. Không tìm được paper ở một venue không có nghĩa venue đó không có related work.

**Đối chiếu phương pháp và trích đoạn abstract** (mỗi nguồn dưới 25 từ trích nguyên văn):
- MPC §3.2 Eq.3–4: caption được mã hóa bởi text encoder CLIP, sau đó trộn với image features. Xem [paper](https://arxiv.org/html/2603.04846v1#S3.SS2) và [captioner của tác giả](https://github.com/LiYuanBoJNU/MPCAttack/blob/main/surrogates/FeatureExtractors/InternVL3_1B.py#L85).
- Grad-ECLIP abstract: “applying channel and spatial weights on token features”. Đây là lý do cần comparator dựa trên token features, không đồng nhất với attention rollout.
- Finer-CAM abstract: “suppresses features shared with other classes and emphasizes the unique, discriminative details”. Suy luận đề xuất: conditioning bằng concept margin có thể giúp chọn vùng phân biệt.
- GDA abstract: “aligns perturbation optimization with text-grounded evidence”. H2 theo nghĩa rộng đã được làm.
- SAGA abstract: “uses a fixed attention map from an open-source LVLM”. Điểm so sánh H3 là fixed map với re-grounding trên ảnh hiện tại, cần kiểm soát compute.

**Kết luận kiểm tra giả thuyết:** không ý nào đủ để khẳng định novelty độc lập. Câu hỏi hẹp để kiểm chứng là: *subject-edited hard negatives có giúp điều khiển concept-margin relevance, và re-grounding có memory dựa trên tiến bộ surrogate có cải thiện targeted transfer khi giữ CLIP/budget cố định không?* Đây là đề xuất của notebook, không phải kết quả có sẵn từ các paper. Memory credit là heuristic, không phải attribution nhân quả. Cần thêm rà soát related work và ASR trên held-out victims trước khi viết novelty claim.

## 2. Objective và ablations

Ảnh $x\in[0,1]$, $x_a=\mathrm{clip}(x_s+\delta,0,1)$, $\|\delta\|_\infty\le16/255$. Các model đều eval/frozen. Với mỗi CLIP $m$, text/image **projected embeddings** cùng dimension và chuẩn hóa trước cosine; FOA patch features chỉ so với patch features của cùng model.

MPC-inspired anchors: $j_s=\mathrm{norm}(\lambda z_s+(1-\lambda)t_s)$ và $j_t=\mathrm{norm}(\lambda z_t+(1-\lambda)t_t)$, $\lambda=0.6$. $L_J=\mathrm{CE}([\omega\cos(z_a,j_t),\cos(z_a,j_s)]/\tau,0)$ với $\omega=2,\tau=0.2$. Minimize CE; upstream MPC trả **negative CE** rồi gradient ascent, cùng hướng gradient.

H1: sửa đúng một noun phrase/subject trong **target caption**, tạo ngân hàng counterfactual text $N$. Chọn top-K bằng mean cosine với $z_a$ trên các CLIP, không dựa vào victim. $L_H=\mathrm{CE}([\cos(z_a,t_t),\cos(z_a,n_1),\ldots]/\tau,0)$. Negative được định nghĩa **so với target mong muốn**, không phải nhãn target để hút ảnh về. Source joint anchor là negative riêng trong $L_J$. Bank cố định trong một mẫu; hard selection cập nhật, không backprop qua top-K/LLM.

$L=\sum_m w_m[aL_{FOA,m}+bL_{J,m}+cL_{H,m}]$, $L_{FOA}=(1-\cos(g_a,g_t))+0.2(1-\mathrm{OTSim}(P_a,P_t))$. Mỗi term trung bình full/crop; target full features cache. $w=\mathrm{softmax}(L_m/(L_{m,prev}+10^{-8}))$ detached, reset từng mẫu. Mỗi model backward riêng rồi ghép gradients, giảm peak VRAM. Các hằng số loss/OT đọc từ FOA/MPC; tổ hợp và weights $a,b,c$ là hyperparameters đề xuất.

H2: rollout của $\cos(z_a,\text{source concept})-\cos(z_a,t_t)$ và hard-negative-vs-target margin. CLIP dual encoder **không có cross-attention text→patch**; token spans chỉ audit BPE của concept, không giả làm attention từ một token LLM. Relevance là gradient-conditioned self-attention. Chọn window tổng relevance cao nhất, giữ area giống random control; chỉ update pixels trong window.

H3: refresh mỗi `REFRESH_EVERY` step trên ảnh đã đổi. `h1_full` thêm EMA và discount vùng vừa cập nhật khi stationary full-image surrogate margin giảm; `h1_dynamic` chỉ fresh map. Credit gán cho các vùng đã dùng trong block, không chứng minh vùng nào là nguyên nhân. Map/negative selection được detach; không second-order gradient.

| Variant | FOA-derived | MPC joint | Negative bank | Region | Temporal memory |
|---|---|---|---|---|---|
| `foa_clip_control` | Có | Không | Không | Random window | Không |
| `mpc_clip_control` | Không | Có | Không | Random window | Không |
| `h1_random` | Có | Có | Hard top-K | Random window | Không |
| `h1_fixed` | Có | Có | Hard top-K | Fixed initial relevance | Không |
| `h1_dynamic` | Có | Có | Hard top-K | Recompute | Không |
| `h1_full` | Có | Có | Hard top-K | Recompute | Có |
| `h1_no_hard` | Có | Có | Không | Recompute | Có |
| `h1_random_neg` | Có | Có | Random K từ cùng bank | Recompute | Có |
| `h1_no_fusion` | Có | Không | Hard top-K | Recompute | Có |
| `h1_no_hard_loss` | Có | Có | Hard top-K cho grounding, tắt hard loss | Recompute | Có |
| `h1_image_anchors` | Có | Có, anchors chỉ từ ảnh | Hard top-K | Recompute | Có |


`h1_full` ↔ `h1_no_hard_loss` tách tác dụng hard loss khỏi việc dùng negatives để grounding. `h1_full` ↔ `h1_image_anchors` giữ joint loss và chỉ đổi image/text fusion; đây là control trực tiếp cho H4. `h1_no_hard` bỏ negatives khỏi cả loss lẫn grounding, đo đóng góp toàn cơ chế H1.

Random control chọn box mới mỗi `REFRESH_EVERY` bước và giữ box trong block, giống nhịp của dynamic variants. Hai RNG riêng cho negative mining và region selection để lựa chọn negatives không làm đổi chuỗi random boxes. Ghi thêm union support qua thời gian: area mỗi bước bằng nhau chưa bảo đảm cumulative support bằng nhau.

Controls là **adapted controls**, không báo là original FOA/MPC: deterministic K-means với gradients qua cluster means, OT plan detached, fixed target anchors, full+crop averaging, support mask, sample-local history và normalizing fused anchors khác upstream. Original FOA baseline nên chạy notebook cũ riêng. `h1_no_fusion` bỏ MPC joint term, vẫn có image/text hard-negative objective; không phải image-only ablation.

**Negative quality:** LLM phải cung cấp span và replacement; code kiểm tra single occurrence, khác positive/source captions, unique, không sửa toàn caption. CLIP filter loại near-duplicate và negative còn hợp target hơn positive. Các kiểm tra này **không chứng nhận counterfactual sai về mặt hình ảnh hoặc loại hết synonyms**; cần review `negative_candidates.json` trước kết luận.

In [ ]:
# Cell 1 — User configuration. No victim query or paid API in this notebook.
from pathlib import Path
DATASET_ROOT = Path('/kaggle/input/datasets/zintom/blackbox-adversarial-attack-dataset')
WORK_ROOT = Path('/kaggle/working')
OUTPUT_ROOT = WORK_ROOT / 'kltn_attack_v4_h1_outputs'
PAIR_MANIFEST = None         # Optional previous attack manifest; preserves its sample_id/index.
SOURCE_DIR = None            # None: discover the unique bigscale directory.
TARGET_DIR = None            # None: discover the unique target_images directory.
NUM_SAMPLES = 1              # 100 or None for the full experiment.
SAMPLE_START = 0
RUN_ATTACK = True
INSTALL_DEPENDENCIES = True
RUN_SELF_CHECKS = True
RESUME_RUN_DIR = None        # Existing writable run; only completed samples are skipped.
RUN_VARIANTS = ['h1_full']   # See ablation table; captions/negative bank shared across variants.
SEED = 2023
ATTACK_DEVICE = 'cuda:0'
CAPTION_DEVICE = 'cuda:0'    # 'cuda:1' if desired; captioner is unloaded before CLIP loading.
BACKBONES = ['B16']     # Smoke test/single GPU. For comparable ensemble experiments, set ['B16','B32'] or restore your full baseline.
GROUNDING_BACKBONE = 'B16'   # Must already be in BACKBONES; no additional vision encoder.
MODEL_REVISION = 'main'     # Resolved to immutable HF commits and saved before loading.
CAPTION_MODEL_ID = 'OpenGVLab/InternVL3-1B-hf'
CAPTION_PROMPT = 'Describe this image in one concise sentence, no longer than 20 words.'
CAPTION_MAX_NEW_TOKENS = 80  # MPC code uses 50; larger cap here, reject truncation.
CONCEPT_MAX_NEW_TOKENS = 192   # Short JSON response, avoid long self-correction loops.
TEXT_RETRIES = 2
MIN_NEGATIVES = 2
MAX_NEGATIVES = 6
MAX_SOURCE_CONCEPTS = 3
MANUAL_TEXT_OVERRIDES = {} # {'00000': {'source': '...', 'target': '...', 'concepts': {...}}}
CONCEPT_FAILURE_POLICY = 'deterministic_fallback' # 'strict' raises; fallback is annotated unverified and MUST be audited before reporting ASR.
NEGATIVE_FILTER_POLICY = 'ranked_relaxation'  # 'strict' for publications; ranked keeps MIN_NEGATIVES when threshold rejects too many.
INPUT_RES = 224
STEPS = 300
EPSILON_255 = 16.0
ALPHA_255 = 1.0
CLUSTER_NUMBER = 10
KMEANS_ITERS = 15
SINKHORN_ITERS = 60
OT_REG = 0.1
FOA_LOCAL_WEIGHT = 0.2
FUSION_LAMBDA = 0.6
TEMPERATURE = 0.2
POSITIVE_OMEGA = 2.0
LOSS_WEIGHTS = {'foa': 1.0, 'joint': 1.0, 'hard': 0.5}
USE_DYNAMIC_ENSEMBLE = True
HARD_TOP_K = 2
MINE_EVERY = 10
REFRESH_EVERY = 30
GROUNDING_LAST_LAYERS = 6
CROP_AREA = 0.5
MEMORY_EMA = 0.7
CREDIT_DECAY = 0.9
CREDIT_STRENGTH = 1.0
NEGATIVE_TEXT_SIM_MAX = 0.98
NEGATIVE_TARGET_MARGIN = 0.0
USE_AUTOCAST = True         # Vision forward only; clustering/OT/loss stay FP32.
GRADIENT_SCALE = 1024.0
MOMENTUM = 0.0              # 0=projected sign descent; MPC momentum 0.9 can be ablated.
LOG_EVERY = 10
SAVE_EVERY = 30
CONTINUE_ON_SAMPLE_ERROR = True
EXPORT_ZIP = True

VARIANTS = {
    'foa_clip_control': dict(foa=True, joint=False, negatives='none', region='random', memory=False),
    'mpc_clip_control': dict(foa=False, joint=True, negatives='none', region='random', memory=False),
    'h1_random': dict(foa=True, joint=True, negatives='hard', region='random', memory=False),
    'h1_fixed': dict(foa=True, joint=True, negatives='hard', region='fixed', memory=False),
    'h1_dynamic': dict(foa=True, joint=True, negatives='hard', region='dynamic', memory=False),
    'h1_full': dict(foa=True, joint=True, negatives='hard', region='dynamic', memory=True),
    'h1_no_hard': dict(foa=True, joint=True, negatives='none', region='dynamic', memory=True),
    'h1_random_neg': dict(foa=True, joint=True, negatives='random', region='dynamic', memory=True),
    'h1_no_fusion': dict(foa=True, joint=False, negatives='hard', region='dynamic', memory=True),
    'h1_no_hard_loss': dict(foa=True, joint=True, negatives='hard', hard_loss=False, region='dynamic', memory=True),
    'h1_image_anchors': dict(foa=True, joint=True, fusion=False, negatives='hard', region='dynamic', memory=True),
}
assert RUN_VARIANTS and len(set(RUN_VARIANTS)) == len(RUN_VARIANTS) and all(v in VARIANTS for v in RUN_VARIANTS)
assert BACKBONES and len(set(BACKBONES)) == len(BACKBONES) and set(BACKBONES) <= {'B16','B32','Laion'}
assert GROUNDING_BACKBONE in BACKBONES
assert NUM_SAMPLES is None or (type(NUM_SAMPLES) is int and NUM_SAMPLES > 0)
assert type(SAMPLE_START) is int and SAMPLE_START >= 0
assert type(STEPS) is int and STEPS > 0 and INPUT_RES == 224
assert set(LOSS_WEIGHTS) == {'foa','joint','hard'} and OT_REG > 0
assert 0 <= NEGATIVE_TEXT_SIM_MAX <= 1 and CREDIT_STRENGTH >= 0 and MAX_SOURCE_CONCEPTS >= 1
assert 0 <= EPSILON_255 <= 255 and ALPHA_255 > 0 and 0 < CROP_AREA <= 1
assert 2 <= CLUSTER_NUMBER <= 49 and 0 <= FUSION_LAMBDA <= 1 and TEMPERATURE > 0
assert 0 <= MOMENTUM < 1 and 0 <= MEMORY_EMA < 1 and 0 <= CREDIT_DECAY <= 1
assert MIN_NEGATIVES >= HARD_TOP_K >= 1 and MAX_NEGATIVES >= MIN_NEGATIVES
assert CONCEPT_FAILURE_POLICY in ('strict','deterministic_fallback')
assert NEGATIVE_FILTER_POLICY in ('strict','ranked_relaxation')
assert all(type(v) is int and v > 0 for v in [TEXT_RETRIES, MINE_EVERY, REFRESH_EVERY, SAVE_EVERY, LOG_EVERY, GROUNDING_LAST_LAYERS, KMEANS_ITERS, SINKHORN_ITERS])
assert all(v >= 0 for v in LOSS_WEIGHTS.values()) and GRADIENT_SCALE > 0


In [ ]:
# Cell 2 — Dependencies. Keep Kaggle Torch/Torchvision; no source checkout required.
import os, sys, subprocess
PACKAGES = ['transformers==4.57.1', 'tokenizers==0.22.1', 'huggingface-hub==0.34.4',
            'accelerate==1.11.0', 'ftfy==6.3.1', 'tqdm==4.67.1', 'matplotlib==3.9.4']
if INSTALL_DEPENDENCIES:
    if any(name in sys.modules for name in ['transformers', 'tokenizers']):
        raise RuntimeError('Restart session before installing packages, or set INSTALL_DEPENDENCIES=False.')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PACKAGES], check=True)
os.environ.update(TOKENIZERS_PARALLELISM='false', WANDB_MODE='disabled')


In [ ]:
# Cell 3 — Imports, reproducibility and trace helpers (no models loaded).
import contextlib, gc, hashlib, io, json, math, random, re, shutil, time
from datetime import datetime, timezone
from importlib import metadata
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
from PIL import Image
from torchvision import transforms
from torchvision.transforms import InterpolationMode
from transformers import CLIPModel, AutoTokenizer, AutoProcessor, AutoModelForImageTextToText
from huggingface_hub import HfApi
from tqdm.auto import tqdm

MODEL_IDS = {'B16': 'openai/clip-vit-base-patch16', 'B32': 'openai/clip-vit-base-patch32',
             'Laion': 'laion/CLIP-ViT-G-14-laion2B-s12B-b42K'}
RUN_DIR, EVENT_PATH = None, None
CODE_SOURCES = {}

def remember_code(name):
    try:
        shell = get_ipython()
    except NameError:
        shell = None
    if shell is not None:
        CODE_SOURCES[name] = shell.history_manager.input_hist_raw[-1]

def json_hash(data):
    return hashlib.sha256(json.dumps(data, sort_keys=True, ensure_ascii=False, allow_nan=False).encode()).hexdigest()

def write_json(path, data):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_name(path.name + '.tmp')
    temp.write_text(json.dumps(data, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')
    temp.replace(path)

def append_jsonl(path, data):
    with Path(path).open('a', encoding='utf-8') as stream:
        stream.write(json.dumps(data, ensure_ascii=False, allow_nan=False) + '\n')
        stream.flush()

def event(stage, action, **fields):
    row = {'utc': datetime.now(timezone.utc).isoformat(), 'stage': stage, 'event': action, **fields}
    print(f"[{row['utc']}] {stage}/{action} | " + ' | '.join(f'{k}={v}' for k,v in fields.items()), flush=True)
    if EVENT_PATH is not None:
        append_jsonl(EVENT_PATH, row)

def read_jsonl(path):
    if not Path(path).exists():
        return []
    # A malformed checkpoint is an error; never silently discard records.
    return [json.loads(line) for line in Path(path).read_text(encoding='utf-8').splitlines() if line.strip()]

def seed_all(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

def sample_seed(sample_id):
    return (SEED + int(hashlib.sha256(sample_id.encode()).hexdigest()[:8], 16)) % (2**31)

def elapsed_text(seconds):
    seconds = int(seconds); return f'{seconds//3600:02d}:{seconds%3600//60:02d}:{seconds%60:02d}'

def save_pixels(path, tensor):
    pixels = (tensor.detach().cpu().squeeze(0).clamp(0,1) * 255).round().to(torch.uint8)
    Image.fromarray(pixels.permute(1,2,0).numpy()).save(path, format='PNG')

def save_adversarial_pixels(path, tensor, source):
    # File pixels, including fractional epsilon settings, must respect the same budget.
    base=(source.detach().cpu().squeeze(0)*255).round().to(torch.int16)
    candidate=(tensor.detach().cpu().squeeze(0).clamp(0,1)*255).round().to(torch.int16)
    radius=math.floor(EPSILON_255+1e-9)
    candidate=torch.maximum(torch.minimum(candidate,base+radius),base-radius).clamp(0,255).to(torch.uint8)
    Image.fromarray(candidate.permute(1,2,0).numpy()).save(path,format='PNG')

def pixel_hash(tensor):
    pixels = (tensor.detach().cpu().clamp(0,1)*255).round().to(torch.uint8)
    return hashlib.sha256(pixels.numpy().tobytes()).hexdigest()

def assert_device(device):
    d = torch.device(device)
    if d.type == 'cuda' and (not torch.cuda.is_available() or (d.index or 0) >= torch.cuda.device_count()):
        raise RuntimeError(f'Unavailable {device}; enable Kaggle GPU or correct the device index.')

remember_code('runtime')


In [ ]:
# Cell 4 — Pair by image identity, never zip two unrelated directory orders.
EXTENSIONS = {'.png','.jpg','.jpeg','.webp','.bmp'}
PREPROCESS = transforms.Compose([transforms.Resize(INPUT_RES, interpolation=InterpolationMode.BICUBIC),
                                transforms.CenterCrop(INPUT_RES), transforms.Lambda(lambda im: im.convert('RGB')),
                                transforms.PILToTensor()])

def load_pixels(path):
    with Image.open(path) as image:
        return PREPROCESS(image).float().unsqueeze(0) / 255.0

def discover_dir(name, override):
    if override is not None:
        path = Path(override)
        if not path.is_dir(): raise FileNotFoundError(path)
        return path.resolve()
    candidates = [p for p in DATASET_ROOT.rglob(name) if p.is_dir()]
    if DATASET_ROOT.name == name: candidates.append(DATASET_ROOT)
    if len(candidates) != 1:
        raise ValueError(f'Expected one {name} directory, found {len(candidates)}. Set SOURCE_DIR/TARGET_DIR explicitly.')
    return candidates[0].resolve()

def image_index(directory):
    index = {}
    for path in sorted(directory.rglob('*')):
        if not path.is_file() or path.suffix.lower() not in EXTENSIONS: continue
        if path.stem in index: raise ValueError(f'Ambiguous image stem: {path.stem} in {directory}')
        index[path.stem] = path.resolve()
    if not index: raise ValueError(f'No images in {directory}')
    return index

def safe_sample_id(value):
    return isinstance(value,str) and re.fullmatch(r'[A-Za-z0-9_-]+', value) is not None

def prepare_pairs():
    if not DATASET_ROOT.is_dir(): raise FileNotFoundError(f'Add Kaggle input and correct DATASET_ROOT: {DATASET_ROOT}')
    sources = image_index(discover_dir('bigscale', SOURCE_DIR))
    targets = image_index(discover_dir('target_images', TARGET_DIR))
    if PAIR_MANIFEST is None:
        if set(sources) != set(targets):
            raise ValueError(f'Unmatched stems: source-only={len(set(sources)-set(targets))}, target-only={len(set(targets)-set(sources))}. Supply an explicit PAIR_MANIFEST if intentional.')
        pairs = [{'index':i,'sample_id':f'{i:05d}','source_path':str(sources[stem]),'target_path':str(targets[stem])}
                 for i,stem in enumerate(sorted(sources))]
    else:
        pairs = []
        for row in read_jsonl(PAIR_MANIFEST):
            source_stem = Path(row['original_path']).stem
            target_stem = Path(row['target_path']).stem
            if source_stem not in sources or target_stem not in targets: raise ValueError('Manifest pair absent from dataset.')
            pairs.append({'index':row['index'],'sample_id':row.get('sample_id',f"{row['index']:05d}"),
                          'source_path':str(sources[source_stem]),'target_path':str(targets[target_stem])})
    if any(type(p['index']) is not int or p['index'] < 0 or not safe_sample_id(p['sample_id']) for p in pairs):
        raise ValueError('Invalid sample index/ID.')
    if len({p['sample_id'] for p in pairs}) != len(pairs) or len({p['index'] for p in pairs}) != len(pairs):
        raise ValueError('Duplicate sample ID/index.')
    pairs = pairs[SAMPLE_START:]
    if NUM_SAMPLES is not None and NUM_SAMPLES > len(pairs): raise ValueError('Requested more pairs than available.')
    pairs = pairs if NUM_SAMPLES is None else pairs[:NUM_SAMPLES]
    if not pairs: raise ValueError('No selected pairs.')
    for pair in pairs:
        pair['raw_sha256'], pair['pixel_sha256'] = {}, {}
        for role in ['source','target']:
            path = Path(pair[role+'_path'])
            pair['raw_sha256'][role] = hashlib.sha256(path.read_bytes()).hexdigest()
            pair['pixel_sha256'][role] = pixel_hash(load_pixels(path))
    return pairs

remember_code('data')


In [ ]:
# Cell 5 — MPC caption model; text-only concept generation is a proposed extension.
def normalize_text(text):
    return re.sub(r'\W+', ' ', text.casefold()).strip()

# An explicit, inspectable fallback: caption-grounded substitutions are NOT
# manually checked counterfactual ground truth. Never treat as validated ASR.
def caption_phrases(caption):
    # Isolate short descriptions after determiners, split at frequent function/verb words.
    breaks = r'\b(?:is|are|was|were|be|being|been|with|without|by|at|on|in|near|next|from|into|while|and then|that|who|which)\b|[,.!?;:]'
    phrases = []
    for chunk in re.split(breaks, caption, flags=re.IGNORECASE):
        chunk = chunk.strip()
        for match in re.finditer(r'\b(?:a|an|the)\s+[\w-]+(?:\s+[\w-]+){0,5}',chunk,flags=re.IGNORECASE):
            phrase = match.group().strip()
            # Avoid article fragments with only a determiner, and duplicate concept spans.
            if len(phrase.split())>=2 and normalize_text(phrase) not in {normalize_text(p) for p in phrases}:
                phrases.append(phrase)
    if not phrases:
        # Language-independent-ish last resort: take a short literal span, not an invented concept.
        match=re.search(r'\S+(?:\s+\S+){0,2}',caption.strip())
        if match: phrases=[match.group().rstrip(',.!?;:')]
    return phrases

# Intended only as a reproducible rescue for a failed SMALL text model.
# Deliberately obvious substitutes keep provenance auditable by humans.
CF_SUBSTITUTES = {
    'dog':['a cat','a panda bear'], 'cat':['a dog','a rabbit'],
    'person':['a robot','a child'], 'man':['a robot','a woman'], 'woman':['a robot','a man'],
    'panda':['a dog','a cat'], 'bear':['a dog','a cat'], 'bird':['a cat','a plane'],
    'car':['a bicycle','a bus'], 'bicycle':['a car','a motorcycle'],
    'toothbrush':['a paintbrush','a spoon'], 'horse':['a zebra','a bicycle'],
    'tree':['a lamp','a building'], 'flower':['a toy','a tree'],
}

def deterministic_counterfactuals(source_caption,target_caption):
    if not source_caption.strip() or not target_caption.strip():
        raise ValueError('Cannot form fallback concepts without both non-empty captions.')
    source_phrases=caption_phrases(source_caption)[:MAX_SOURCE_CONCEPTS]
    target_phrases=caption_phrases(target_caption)
    source_words=re.findall(r'\b[\w-]+\b',source_caption)
    if not source_phrases and source_words:
        source_phrases=[' '.join(source_words[:min(3,len(source_words))])]
    # Extract a second editable span if there are no determiners.
    if len(target_phrases)<2:
        target_words=re.findall(r'\b[\w-]{4,}\b',target_caption)
        ignore={'this','that','with','from','into','being','there','which','while','they','have','been'}
        target_phrases += [w for w in target_words if w.casefold() not in ignore and
                           w.casefold() not in {p.casefold() for p in target_phrases}]
    target_phrases=[p for p in target_phrases if p and len(p.split())<=6 and
                    len(re.findall(r'(?<!\w)'+re.escape(p)+r'(?!\w)',target_caption,flags=re.I))==1]
    # Preference: source caption contrast, then simple unambiguous substitutions.
    generic=['a bicycle','a cat','a dog','a robot','a tree','a bus','a chair','a bird','a boat','a horse']
    edits=[];seen=set()
    per_span_cap=max(2, math.ceil(MAX_NEGATIVES/max(1,len(target_phrases))))
    for phrase in target_phrases:
        added_for_span=0
        tokens=normalize_text(phrase).split()
        substitutions=[]
        for word in reversed(tokens):
            substitutions.extend(CF_SUBSTITUTES.get(word,[]))
        substitutions += source_phrases + generic
        for substitution in substitutions:
            if normalize_text(phrase)==normalize_text(substitution): continue
            # Reject edits whose main content head (e.g. dog -> a dog) is unchanged.
            head=tokens[-1] if tokens else ''
            if head and head in normalize_text(substitution).split(): continue
            replacement=substitution.strip()
            pattern=re.compile(r'(?<!\w)'+re.escape(phrase)+r'(?!\w)',re.IGNORECASE)
            candidate=pattern.sub(replacement,target_caption,count=1)
            key=normalize_text(candidate)
            if key in seen or key in (normalize_text(source_caption),normalize_text(target_caption)): continue
            seen.add(key);edits.append({'span':phrase,'replacement':replacement})
            added_for_span+=1
            if added_for_span>=per_span_cap or len(edits)>=MAX_NEGATIVES: break
        if len(edits)>=MAX_NEGATIVES: break
    parsed=parse_concepts({'source_concepts':source_phrases,'target_edits':edits},source_caption,target_caption)
    parsed['provenance']='deterministic_fallback_unverified'
    parsed['requires_manual_review']=True
    return parsed

def parse_concepts(raw, source_caption, target_caption):
    if isinstance(raw,str):
        left, right = raw.find('{'), raw.rfind('}')
        if left < 0 or right < left: raise ValueError('Concept response needs one JSON object.')
        raw = json.loads(raw[left:right+1])
    if not isinstance(raw,dict): raise ValueError('Concept output must be an object.')
    source_concepts = raw.get('source_concepts')
    edits = raw.get('target_edits')
    if not isinstance(source_concepts,list) or not isinstance(edits,list): raise ValueError('Missing concept lists.')
    concepts = []
    for phrase in source_concepts:
        if not isinstance(phrase,str) or not phrase.strip() or phrase.casefold() not in source_caption.casefold():
            raise ValueError('Source concept must be a literal nonempty caption span.')
        if normalize_text(phrase) not in {normalize_text(p) for p in concepts}: concepts.append(phrase.strip())
    if not concepts: raise ValueError('At least one valid source concept is required.')
    candidates, rejected, seen = [], [], set()
    for edit in edits:
        if not isinstance(edit,dict): rejected.append({'edit':edit,'reason':'invalid_object'}); continue
        span, replacement = edit.get('span'), edit.get('replacement')
        if not isinstance(span,str) or not isinstance(replacement,str) or not span.strip() or not replacement.strip():
            rejected.append({'edit':edit,'reason':'empty_span'}); continue
        # Word boundaries keep 'cat' from rewriting 'cattle'. Exactly one occurrence.
        pattern = re.compile(r'(?<!\w)' + re.escape(span.strip()) + r'(?!\w)', re.IGNORECASE)
        matches = list(pattern.finditer(target_caption))
        if len(matches) != 1 or normalize_text(span) == normalize_text(replacement) or len(span.split()) > 6:
            rejected.append({'edit':edit,'reason':'not_single_concept_edit'}); continue
        if normalize_text(span) == normalize_text(target_caption):
            rejected.append({'edit':edit,'reason':'whole_caption_rewrite'}); continue
        counterfactual = pattern.sub(lambda _: replacement.strip(),target_caption,count=1)
        key = normalize_text(counterfactual)
        if key in seen or key in {normalize_text(source_caption),normalize_text(target_caption)}:
            rejected.append({'edit':edit,'reason':'duplicate_or_positive'}); continue
        seen.add(key)
        candidates.append({'span':matches[0].group(), 'replacement':replacement.strip(), 'caption':counterfactual,
                           'semantic_falsehood_verified':False})
    if len(candidates) < MIN_NEGATIVES:
        raise ValueError(f'Need {MIN_NEGATIVES} valid single-span negatives; obtained {len(candidates)}.')
    return {'source_concepts':concepts[:MAX_SOURCE_CONCEPTS], 'negative_candidates':candidates[:MAX_NEGATIVES],
            'rejected_edits':rejected, 'raw_generation':raw}

class MPCCaptioner:
    def __init__(self, revision):
        assert_device(CAPTION_DEVICE)
        device=torch.device(CAPTION_DEVICE)
        self.dtype = (torch.bfloat16 if torch.cuda.get_device_capability(device)[0] >= 8 else torch.float16) if device.type=='cuda' else torch.float32
        self.processor = AutoProcessor.from_pretrained(CAPTION_MODEL_ID, revision=revision, trust_remote_code=False)
        self.model = AutoModelForImageTextToText.from_pretrained(CAPTION_MODEL_ID, revision=revision,
                     torch_dtype=self.dtype, low_cpu_mem_usage=True, trust_remote_code=False).eval().requires_grad_(False).to(CAPTION_DEVICE)

    @torch.inference_mode()
    def generate(self, prompt, image=None, limit=CAPTION_MAX_NEW_TOKENS):
        content = ([{'type':'image'}] if image is not None else []) + [{'type':'text','text':prompt}]
        messages = [{'role':'user','content':content}]
        rendered = self.processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
        options = dict(text=rendered, return_tensors='pt')
        if image is not None: options['images'] = [image]
        inputs = self.processor(**options).to(self.model.device, dtype=self.dtype)
        tokens = self.model.generate(**inputs, do_sample=False, max_new_tokens=limit)
        generated = tokens[0, inputs['input_ids'].shape[1]:]
        if len(generated) >= limit: raise ValueError('Caption/concept generation hit max_new_tokens; increase the cap in a new run.')
        text = self.processor.decode(generated, skip_special_tokens=True).strip()
        if not text: raise ValueError('Empty caption/concept response.')
        return text

    def concepts(self, source, target, sample_id):
        # The original 1B model sometimes returns prose or malformed JSON.
        # Use a short, fixed-schema prompt and avoid concatenating failed responses forever.
        base_prompt = (
            'Return exactly one compact JSON object, without markdown or explanation.\n'
            'Schema: {"source_concepts":["exact phrase from source"],'
            '"target_edits":[{"span":"exact phrase from target","replacement":"wrong object"},'
            '{"span":"exact phrase from target","replacement":"another wrong object"}]}.\n'
            f'Copy 1-{MAX_SOURCE_CONCEPTS} source noun phrases; make {MAX_NEGATIVES} target edits. '
            'Each span appears EXACTLY once in target and has at most 6 words. '
            'Change only that span, not the full sentence. Wrong object, not synonym.\n'
            f'Source: {source}\nTarget: {target}\nJSON:')
        errors=[]
        for attempt in range(1,TEXT_RETRIES+1):
            event('concept','generate',sample_id=sample_id,attempt=attempt)
            try:
                prompt=base_prompt if attempt==1 else (base_prompt+'\nRemember: valid JSON and exact copied spans only.')
                raw=self.generate(prompt, limit=CONCEPT_MAX_NEW_TOKENS)
                result=parse_concepts(raw,source,target)
                result['provenance']='caption_model_generated_unverified'
                result['requires_manual_review']=True
                event('concept','valid',sample_id=sample_id,negatives=len(result['negative_candidates']),source_concepts=result['source_concepts'])
                return result
            except (ValueError, json.JSONDecodeError) as error:
                errors.append(f'{type(error).__name__}: {error}')
                event('concept','invalid',sample_id=sample_id,attempt=attempt,error=str(error))
        if CONCEPT_FAILURE_POLICY == 'strict':
            raise ValueError('Concept generation failed after '
                             f'{TEXT_RETRIES} attempts: {errors}. Configure MANUAL_TEXT_OVERRIDES.')
        try:
            result=deterministic_counterfactuals(source,target)
        except ValueError as error:
            raise ValueError(f'Model and deterministic concept generation both failed: {errors}; {error}') from error
        result['model_generation_errors']=errors
        event('concept','fallback_used',sample_id=sample_id,negatives=len(result['negative_candidates']),
              provenance=result['provenance'],manual_review_required=True)
        return result

remember_code('caption_and_concepts')


In [ ]:
# Cell 6 — CLIP adapters: FOA hidden features + projected image/text for MPC.
class ClipBackbone(nn.Module):
    def __init__(self,name,revision,model=None,tokenizer=None,device=ATTACK_DEVICE):
        super().__init__(); self.name=name; self.device=torch.device(device)
        self.model = model if model is not None else CLIPModel.from_pretrained(MODEL_IDS[name],revision=revision,attn_implementation='eager',torch_dtype=torch.float32)
        self.tokenizer = tokenizer if tokenizer is not None else AutoTokenizer.from_pretrained(MODEL_IDS[name],revision=revision)
        self.calls={'vision_forward':0,'attention_forward':0,'text_forward':0,'attack_backward':0,'attribution_backward':0}
        self.model.eval().requires_grad_(False)
        self.model.vision_model.to(self.device); self.model.visual_projection.to(self.device)
        # Text weights stay on CPU; text features are cached once per sample and then copied.
        self.model.text_model.cpu(); self.model.text_projection.cpu()
        self.mean = torch.tensor([0.48145466,0.4578275,0.40821073],device=self.device).view(1,3,1,1)
        self.std = torch.tensor([0.26862954,0.26130258,0.27577711],device=self.device).view(1,3,1,1)

    def features(self,x,attentions=False):
        self.calls['vision_forward']+=1
        self.calls['attention_forward']+=int(attentions)
        pixel_values=(x.clamp(0,1)-self.mean)/self.std
        use_amp = USE_AUTOCAST and self.device.type=='cuda' and not attentions
        with torch.autocast(device_type=self.device.type,dtype=torch.float16,enabled=use_amp):
            out=self.model.vision_model(pixel_values=pixel_values,output_attentions=attentions)
            projected=self.model.visual_projection(out.pooler_output)
        features=out.last_hidden_state.float()
        return {'joint':F.normalize(projected.float(),dim=-1),'global':F.normalize(features[:,0],dim=-1),
                'patches':F.normalize(features[:,1:],dim=-1),'attentions':out.attentions if attentions else None}

    @torch.no_grad()
    def text_features(self,texts):
        self.calls['text_forward']+=1
        max_length=self.model.config.text_config.max_position_embeddings
        tokens=self.tokenizer(texts,padding=True,truncation=False,return_tensors='pt')
        if tokens['input_ids'].shape[1]>max_length: raise ValueError(f'{self.name}: text exceeds CLIP context; no silent truncation.')
        out=self.model.text_model(**tokens)
        vectors=F.normalize(self.model.text_projection(out.pooler_output).float(),dim=-1)
        return vectors.to(self.device)

    def token_spans(self,caption,phrases):
        # Offsets audit tokenization only; no direct CLIP text-to-image attention exists.
        if not self.tokenizer.is_fast: return {'available':False,'reason':'slow tokenizer'}
        encoded=self.tokenizer(caption,return_offsets_mapping=True,truncation=False)
        records=[]
        for phrase in phrases:
            start=caption.casefold().find(phrase.casefold());end=start+len(phrase)
            positions=[i for i,(a,b) in enumerate(encoded['offset_mapping']) if b>a and a<end and b>start]
            records.append({'phrase':phrase,'characters':[start,end],'token_positions':positions,
                            'tokens':self.tokenizer.convert_ids_to_tokens([encoded['input_ids'][i] for i in positions])})
        return {'available':True,'phrases':records}

def cluster_patches(patches,k=CLUSTER_NUMBER):
    if patches.ndim!=2 or patches.shape[0]<k or not torch.isfinite(patches).all(): raise ValueError('Invalid K-means patches.')
    # Deterministic farthest-point initialization; assignments detached, means differentiable.
    with torch.no_grad():
        x=patches.detach().float(); chosen=[0]; distance=torch.full((len(x),),float('inf'),device=x.device)
        for _ in range(1,k):
            distance=torch.minimum(distance,((x-x[chosen[-1]])**2).sum(-1));chosen.append(int(distance.argmax()))
        centers=x[chosen].clone()
        for _ in range(KMEANS_ITERS):
            labels=torch.cdist(x,centers).argmin(-1)
            sums=torch.zeros_like(centers).index_add_(0,labels,x)
            counts=torch.bincount(labels,minlength=k).to(x.dtype).unsqueeze(1)
            updated=torch.where(counts>0,sums/counts.clamp_min(1),centers)
            if torch.allclose(updated,centers,atol=1e-5): centers=updated;break
            centers=updated
        labels=torch.cdist(x,centers).argmin(-1)
        counts=torch.bincount(labels,minlength=k).to(x.dtype).unsqueeze(1)
    sums=patches.new_zeros((k,patches.shape[-1])).index_add(0,labels,patches)
    fallback=patches[chosen]
    return torch.where(counts>0,sums/counts.clamp_min(1),fallback)

def ot_similarity(first,second):
    sim=F.normalize(first.float(),dim=-1) @ F.normalize(second.float(),dim=-1).T
    with torch.no_grad():
        log_kernel=(sim.detach()-1)/OT_REG
        u=torch.zeros(len(first),device=sim.device);v=torch.zeros(len(second),device=sim.device)
        for _ in range(SINKHORN_ITERS):
            u=-math.log(len(first))-torch.logsumexp(log_kernel+v[None,:],dim=1)
            v=-math.log(len(second))-torch.logsumexp(log_kernel+u[:,None],dim=0)
        plan=torch.exp(log_kernel+u[:,None]+v[None,:])
    score=(plan*sim).sum()
    if not torch.isfinite(score): raise FloatingPointError('Non-finite OT score.')
    return score

def anchor_ce(z,positive,negatives,omega=1.0):
    pos=omega*(z*positive).sum(-1,keepdim=True)
    neg=z@negatives.T
    logits=torch.cat([pos,neg],dim=1)/TEMPERATURE
    return F.cross_entropy(logits,torch.zeros(len(z),dtype=torch.long,device=z.device))

@torch.no_grad()
def build_anchors(models,source,target,text_record):
    result={};texts=[text_record['source'],text_record['target']]+text_record['concepts']['source_concepts']
    texts += [n['caption'] for n in text_record['concepts']['negative_candidates']]
    concept_count=len(text_record['concepts']['source_concepts'])
    for name,model in models.items():
        encoded=model.text_features(texts)
        s=model.features(source);t=model.features(target)
        result[name]={'source_joint':F.normalize(FUSION_LAMBDA*s['joint']+(1-FUSION_LAMBDA)*encoded[0:1],dim=-1),
                      'target_joint':F.normalize(FUSION_LAMBDA*t['joint']+(1-FUSION_LAMBDA)*encoded[1:2],dim=-1),
                      'source_text':encoded[0:1],'target_text':encoded[1:2],
                      'source_concepts':encoded[2:2+concept_count],'negative_texts':encoded[2+concept_count:],
                      'target_global':t['global'],'target_centers':cluster_patches(t['patches'][0]),
                      'target_image':t['joint'],'source_image':s['joint']}
    # Use the configured grounding CLIP to filter bank before any variant sees it.
    probe=result[GROUNDING_BACKBONE];neg=probe['negative_texts'];positive=probe['target_text']
    text_sim=(neg@positive.T).squeeze(1)
    clean_sim=(neg@probe['target_image'].T).squeeze(1)
    positive_sim=(probe['target_image']*positive).sum()
    strict_mask=(text_sim<NEGATIVE_TEXT_SIM_MAX)&(clean_sim<=positive_sim-NEGATIVE_TARGET_MARGIN)
    strict_keep=strict_mask.nonzero().flatten()
    keep=strict_keep
    relaxed=False
    if len(keep)<MIN_NEGATIVES:
        if NEGATIVE_FILTER_POLICY=='strict':
            raise ValueError(f'Only {len(keep)} negatives survive CLIP filter; review captions/edits or select ranked_relaxation for exploratory runs.')
        # Keep the best-fitting generated edits rather than abort the entire attack.
        # This changes the filter relative to the original hypothesis; report it.
        penalty=(text_sim-NEGATIVE_TEXT_SIM_MAX).clamp_min(0)+(clean_sim-positive_sim+NEGATIVE_TARGET_MARGIN).clamp_min(0)
        candidates=[i for i in range(len(text_sim)) if i not in strict_keep.tolist()]
        candidates.sort(key=lambda i:(float(penalty[i]),i))
        chosen=strict_keep.tolist()+candidates[:max(0,MIN_NEGATIVES-len(strict_keep))]
        keep=torch.tensor(chosen,device=text_sim.device,dtype=torch.long)
        relaxed=True
        event('concept','negative_filter_relaxed',strict_survivors=len(strict_keep),
              chosen_indices=chosen,policy=NEGATIVE_FILTER_POLICY)
    audit={'candidate_indices':keep.cpu().tolist(),'strict_candidate_indices':strict_keep.cpu().tolist(),
           'negative_filter_relaxed':relaxed,'negative_filter_policy':NEGATIVE_FILTER_POLICY,
           'text_similarity':text_sim.cpu().tolist(),
           'target_image_similarity':clean_sim.cpu().tolist(),'positive_image_similarity':float(positive_sim),
           'semantic_falsehood_verified':False,'requires_manual_review':True}
    for anchor in result.values(): anchor['negative_texts']=anchor['negative_texts'][keep]
    return result,audit

remember_code('clip_and_losses')


In [ ]:
# Cell 7 — Text-conditioned CLIP relevance; no cross-attention or victim access.
def attention_rollout(attentions,gradients):
    count=attentions[0].shape[-1]
    relevance=torch.eye(count,device=attentions[0].device,dtype=torch.float32).unsqueeze(0)
    for attention,gradient in zip(attentions,gradients):
        cam=(attention.float()*gradient.float()).clamp_min(0).mean(dim=1).detach()
        relevance=relevance+cam@relevance
    patches=relevance[:,0,1:]
    side=math.isqrt(patches.shape[-1])
    if side*side!=patches.shape[-1]: raise ValueError('Non-square CLIP patch grid.')
    return patches.reshape(1,1,side,side)

def probability_map(value):
    value=torch.nan_to_num(value.detach().float(),nan=0,posinf=0,neginf=0).clamp_min(0)
    mass=value.sum()
    return (value/mass if mass>1e-12 else torch.full_like(value,1/value.numel())),bool(mass<=1e-12)

@torch.no_grad()
def mine_negatives(models,anchors,image,mode,rng):
    if mode=='none': return [],[]
    scores=[]
    for name,model in models.items():
        z=model.features(image)['joint'];scores.append((z@anchors[name]['negative_texts'].T).squeeze(0))
    mean=torch.stack(scores).mean(0)
    k=min(HARD_TOP_K,len(mean))
    indices=torch.randperm(len(mean),generator=rng)[:k] if mode=='random' else mean.topk(k).indices.cpu()
    return indices.tolist(),mean.cpu().tolist()

class ConceptGrounder:
    def __init__(self,model,anchor,variant):
        self.model=model;self.anchor=anchor;self.variant=variant
        self.ema=None;self.credit=None;self.visits=None;self.previous_margin=None;self.priority=None

    def note_region(self,box):
        if self.visits is not None:
            top,left,height,width=box;self.visits[:,:,top:top+height,left:left+width]+=1

    def refresh(self,image,negative_ids):
        x=image.detach().clone().requires_grad_(True)
        with torch.enable_grad():
            features=self.model.features(x,attentions=True);z=features['joint']
            # Ground source evidence plus active confusing counterfactuals versus desired target.
            candidates=self.anchor['source_concepts']
            if negative_ids:
                candidates=torch.cat([candidates,self.anchor['negative_texts'][negative_ids]],dim=0)
            margins=(z@candidates.T).flatten()-(z*self.anchor['target_text']).sum()
            all_neg=torch.cat([self.anchor['source_concepts'],self.anchor['negative_texts']],dim=0)
            stationary_margin=float(((z@all_neg.T).max()-(z*self.anchor['target_text']).sum()).detach())
            attentions=features['attentions'][-GROUNDING_LAST_LAYERS:]
            if not attentions or any(a is None for a in attentions): raise RuntimeError('CLIP eager attention tensors unavailable.')
            maps=[];fallbacks=0
            selected_labels=list(range(len(self.anchor['source_concepts'])))+list(negative_ids)
            weights=torch.softmax(margins.detach()/TEMPERATURE,dim=0)
            for i,margin in enumerate(margins):
                gradients=torch.autograd.grad(margin,attentions,retain_graph=i<len(margins)-1,allow_unused=False)
                self.model.calls['attribution_backward']+=1
                relevance=attention_rollout(attentions,gradients)
                relevance=F.interpolate(relevance,size=image.shape[-2:],mode='bilinear',align_corners=False)
                prob,fallback=probability_map(relevance);maps.append(prob);fallbacks+=int(fallback)
            fresh,_=probability_map(sum(w*m for w,m in zip(weights,maps)))
        progress=0.0
        if self.ema is None:
            self.ema=fresh;self.credit=torch.zeros_like(fresh);self.visits=torch.zeros_like(fresh)
        elif self.variant['memory']:
            progress=max(0.0,self.previous_margin-stationary_margin)
            visit_distribution=self.visits/self.visits.sum().clamp_min(1)
            self.credit=CREDIT_DECAY*self.credit+progress*visit_distribution
            self.ema=MEMORY_EMA*self.ema+(1-MEMORY_EMA)*fresh
        else:
            self.ema=fresh
        self.last_concept_maps=torch.cat(maps,dim=0).detach().cpu()
        self.visits.zero_();self.previous_margin=stationary_margin
        normalized_credit=self.credit/self.credit.max().clamp_min(1e-12)
        self.priority,_=probability_map(self.ema/(1+CREDIT_STRENGTH*normalized_credit) if self.variant['memory'] else fresh)
        return self.priority,{'stationary_margin':stationary_margin,'block_margin_improvement':progress,
                              'source_concept_count':len(self.anchor['source_concepts']),'concept_order':selected_labels,
                              'concept_margins':margins.detach().cpu().tolist(),'concept_weights':weights.cpu().tolist(),
                              'uniform_fallback_maps':fallbacks,'active_negative_ids':negative_ids}

def crop_shape(height,width):
    side=max(1,min(height,width,round(math.sqrt(CROP_AREA*height*width))))
    return side,side

def choose_box(probability,image_shape,rng,random_box=False):
    height,width=image_shape[-2:];h,w=crop_shape(height,width)
    if random_box:
        top=int(torch.randint(height-h+1,(1,),generator=rng));left=int(torch.randint(width-w+1,(1,),generator=rng))
    else:
        mass=F.avg_pool2d(probability,kernel_size=(h,w),stride=1)
        flat=int(mass.flatten().argmax());top,left=divmod(flat,width-w+1)
    return [top,left,h,w]

def crop_view(image,box):
    top,left,height,width=box
    return F.interpolate(image[:,:,top:top+height,left:left+width],size=image.shape[-2:],mode='bilinear',align_corners=False)

def support_mask(image,box):
    top,left,height,width=box;mask=torch.zeros_like(image[:,:1])
    mask[:,:,top:top+height,left:left+width]=1
    return mask

remember_code('grounding_and_regions')


In [ ]:
# Cell 8 — One attack sample. Minimize all distances / CE objectives.
def objective_terms(model,anchor,image,variant,negative_ids):
    feat=model.features(image);z=feat['joint'];zero=z.sum()*0
    terms={'foa':zero,'joint':zero,'hard':zero}
    if variant['foa']:
        global_distance=1-(feat['global']*anchor['target_global']).sum(-1).mean()
        local_distance=1-ot_similarity(cluster_patches(feat['patches'][0]),anchor['target_centers'])
        terms['foa']=global_distance+FOA_LOCAL_WEIGHT*local_distance
    if variant['joint']:
        positive=anchor['target_joint'] if variant.get('fusion',True) else anchor['target_image']
        negative=anchor['source_joint'] if variant.get('fusion',True) else anchor['source_image']
        terms['joint']=anchor_ce(z,positive,negative,omega=POSITIVE_OMEGA)
    if variant['negatives']!='none' and variant.get('hard_loss',True):
        if not negative_ids: raise ValueError('Hard-negative term requires selected negative IDs.')
        terms['hard']=anchor_ce(z,anchor['target_text'],anchor['negative_texts'][negative_ids])
    return terms

def project_image(candidate,source):
    epsilon=EPSILON_255/255.0
    return torch.maximum(torch.minimum(candidate,source+epsilon),source-epsilon).clamp(0,1)

def run_sample(models,anchors,source,target,text_record,variant_name,directory,on_step=None):
    variant=VARIANTS[variant_name];directory=Path(directory)
    directory.mkdir(parents=True,exist_ok=True);(directory/'maps').mkdir(exist_ok=True)
    rng=torch.Generator(device='cpu').manual_seed(sample_seed(text_record['sample_id']))
    region_rng=torch.Generator(device='cpu').manual_seed(sample_seed(text_record['sample_id']))
    call_start={name:dict(model.calls) for name,model in models.items()}
    if source.device.type=='cuda':
        torch.cuda.synchronize(source.device);torch.cuda.reset_peak_memory_stats(source.device)
    visited_support=torch.zeros_like(source[:,:1])
    box=None
    image=source.detach().clone();momentum=torch.zeros_like(image)
    previous_losses=None;negative_ids=[];negative_scores=[];grounder=None
    if variant['region']!='random': grounder=ConceptGrounder(models[GROUNDING_BACKBONE],anchors[GROUNDING_BACKBONE],variant)
    # Restart partial sample: step trace reflects only this invocation, events preserve history.
    trace_path=directory/'steps.jsonl';trace_path.write_text('',encoding='utf-8')
    started=time.perf_counter()
    bar=tqdm(total=STEPS,desc=f"{variant_name}/{text_record['sample_id']}",unit='step')
    try:
        for step in range(STEPS):
            tick=time.perf_counter()
            if step%MINE_EVERY==0:
                negative_ids,negative_scores=mine_negatives(models,anchors,image,variant['negatives'],rng)
            refresh=grounder is not None and (step==0 or (variant['region']=='dynamic' and step%REFRESH_EVERY==0))
            if refresh:
                probability,grounding=grounder.refresh(image,negative_ids)
                box=choose_box(probability,image.shape,rng)
                np.save(directory/'maps'/f'relevance_{step:05d}.npy',probability.cpu().numpy())
                np.save(directory/'maps'/f'concept_maps_{step:05d}.npy',grounder.last_concept_maps.numpy())
                write_json(directory/'maps'/f'relevance_{step:05d}.json',{'computed_after_steps':step,'box':box,**grounding})
                save_pixels(directory/'maps'/f'image_{step:05d}.png',image)
                event('grounding','refresh',sample_id=text_record['sample_id'],variant=variant_name,step=step,
                      box=box,margin=round(grounding['stationary_margin'],5),fallbacks=grounding['uniform_fallback_maps'])
            if grounder is None and (step==0 or step%REFRESH_EVERY==0):
                box=choose_box(None,image.shape,region_rng,random_box=True)
            grads=[];losses=[];details={}
            for name,model in models.items():
                x=image.detach().requires_grad_(True)
                full=objective_terms(model,anchors[name],x,variant,negative_ids)
                local=objective_terms(model,anchors[name],crop_view(x,box),variant,negative_ids)
                terms={key:(full[key]+local[key])/2 for key in full}
                loss=sum(LOSS_WEIGHTS[key]*terms[key] for key in terms)
                if not torch.isfinite(loss): raise FloatingPointError(f'Non-finite loss: {name}')
                grad=torch.autograd.grad(loss*GRADIENT_SCALE,x)[0]/GRADIENT_SCALE
                model.calls['attack_backward']+=1
                if not torch.isfinite(grad).all(): raise FloatingPointError(f'Non-finite gradient: {name}')
                grads.append(grad.detach());losses.append(loss.detach().float())
                details[name]={key:float(value.detach()) for key,value in terms.items()}
                del x,full,local,terms,loss,grad
            current_losses=torch.stack(losses)
            if USE_DYNAMIC_ENSEMBLE and previous_losses is not None:
                ratios=current_losses/previous_losses.clamp_min(1e-8)
                weights=torch.softmax(ratios,dim=0)
            else: weights=torch.full_like(current_losses,1/len(current_losses))
            previous_losses=current_losses
            gradient=sum(weight*g for weight,g in zip(weights,grads))
            mask=support_mask(image,box)
            visited_support=torch.maximum(visited_support,mask)
            # Mask BEFORE and AFTER momentum, so previous regions cannot leak updates.
            gradient=gradient*mask
            scale=gradient.abs().mean().clamp_min(1e-12)
            momentum=(MOMENTUM*momentum+gradient/scale)*mask
            with torch.no_grad(): image=project_image(image-(ALPHA_255/255)*momentum.sign(),source)
            if grounder is not None: grounder.note_region(box)
            row={'step':step+1,'objective':float((weights*current_losses).sum()),'per_model':details,
                 'ensemble_weights':dict(zip(models,weights.cpu().tolist())), 'box':box,'negative_ids':negative_ids,
                 'negative_similarities':negative_scores,'linf_255':float((image-source).abs().max()*255),
                 'step_support_fraction':float(mask.mean()),'union_support_fraction':float(visited_support.mean()),
                 'seconds':time.perf_counter()-tick,'grounding_refreshed':refresh}
            append_jsonl(trace_path,row);bar.update(1)
            if (step+1)%LOG_EVERY==0 or step==0 or step+1==STEPS:
                elapsed=time.perf_counter()-started;eta=elapsed/(step+1)*(STEPS-step-1)
                bar.set_postfix(loss=round(row['objective'],4),linf=round(row['linf_255'],2),eta=elapsed_text(eta))
                event('attack','progress',sample_id=text_record['sample_id'],variant=variant_name,steps=f'{step+1}/{STEPS}',
                      loss=round(row['objective'],5),linf_255=round(row['linf_255'],3),box=box,negatives=negative_ids,eta=elapsed_text(eta))
            if (step+1)%SAVE_EVERY==0 or step+1==STEPS:
                save_pixels(directory/f'checkpoint_{step+1:05d}.png',image)
            if on_step is not None: on_step(step+1,row)
        if float((image-source).abs().max()*255)>EPSILON_255+1e-3: raise AssertionError('L-infinity budget violated.')
        if source.device.type=='cuda': torch.cuda.synchronize(source.device)
        counts={name:{key:value-call_start[name][key] for key,value in model.calls.items()} for name,model in models.items()}
        return image.detach(),{'steps':STEPS,'seconds':time.perf_counter()-started,'final_surrogate_loss':row['objective'],
                              'compute_counts':counts,'union_support_fraction':float(visited_support.mean()),
                              'peak_cuda_allocated_gib':torch.cuda.max_memory_allocated(source.device)/2**30 if source.device.type=='cuda' else None}
    finally:
        bar.close()

remember_code('optimizer')


In [ ]:
# Cell 9 — Cheap checks; no downloads, captioner or CLIP weights required.
def self_checks():
    x=torch.randn(20,8,requires_grad=True)
    centers=cluster_patches(x,k=4)
    score=ot_similarity(centers,cluster_patches(torch.randn(20,8),k=4))
    gradient=torch.autograd.grad(score,x)[0]
    assert torch.isfinite(gradient).all() and gradient.abs().sum()>0
    source=torch.rand(1,3,16,16);candidate=source+torch.randn_like(source)
    projected=project_image(candidate,source)
    assert float((projected-source).abs().max())<=EPSILON_255/255+1e-6 and 0<=projected.min()<=projected.max()<=1
    raw={'source_concepts':['a dog'],'target_edits':[{'span':'a cat','replacement':f'a replacement{i}'} for i in range(max(MIN_NEGATIVES,2))]}
    parsed=parse_concepts(raw,'a dog on grass','a cat on grass')
    assert parsed['negative_candidates'][0]['caption']=='a replacement0 on grass'
    recovered=deterministic_counterfactuals(
        'A panda bear with a black face and white patches is grooming itself.',
        'A black and white dog is being brushed by a person with a red toothbrush.')
    assert len(recovered['negative_candidates'])>=MIN_NEGATIVES
    assert recovered['provenance']=='deterministic_fallback_unverified'
    assert all(n['caption'] != 'A black and white dog is being brushed by a person with a red toothbrush.'
               for n in recovered['negative_candidates'])
    p=torch.zeros(1,1,16,16);p[:,:,8:,8:]=1;p,_=probability_map(p)
    box=choose_box(p,source.shape,torch.Generator().manual_seed(0))
    mask=support_mask(source,box)
    assert mask.sum()==box[2]*box[3] and box[0]+box[2]<=16 and box[1]+box[3]<=16
    z=torch.tensor([[0.,1.]],requires_grad=True)
    loss=anchor_ce(z,torch.tensor([[1.,0.]]),torch.tensor([[0.,1.]]))
    g=torch.autograd.grad(loss,z)[0]
    assert g[0,0]<0 and g[0,1]>0 # gradient descent attracts target and repels negative.
    event('checks','passed',checks='K-means/OT gradients, projection, JSON edits, incident fallback, region bounds, contrastive sign')

def clip_architecture_check():
    # Random tiny HF CLIP: verify the real APIs/attention gradient graph without downloading weights.
    from transformers import CLIPConfig, CLIPVisionConfig, CLIPTextConfig
    tc=CLIPTextConfig(vocab_size=300,hidden_size=32,intermediate_size=48,num_hidden_layers=2,
                      num_attention_heads=4,max_position_embeddings=77,eos_token_id=299)
    vc=CLIPVisionConfig(hidden_size=32,intermediate_size=48,num_hidden_layers=2,
                        num_attention_heads=4,image_size=INPUT_RES,patch_size=32)
    config=CLIPConfig(text_config=tc.to_dict(),vision_config=vc.to_dict(),projection_dim=16)
    config._attn_implementation='eager'
    class TinyTokenizer:
        def __call__(self,*args,**kwargs):
            return {'input_ids':torch.tensor([[1,7,299]]),'attention_mask':torch.ones(1,3,dtype=torch.long)}
    model=ClipBackbone('B16','random-test',model=CLIPModel(config),tokenizer=TinyTokenizer(),device='cpu')
    text=model.text_features(['test concept'])
    image=torch.rand(1,3,INPUT_RES,INPUT_RES,requires_grad=True)
    features=model.features(image,attentions=True)
    score=(features['joint']*text).sum()
    gradients=torch.autograd.grad(score,features['attentions'],retain_graph=True)
    input_gradient=torch.autograd.grad(score,image)[0]
    relevance=attention_rollout(features['attentions'],gradients)
    assert relevance.shape==(1,1,7,7) and torch.isfinite(relevance).all()
    assert torch.isfinite(input_gradient).all() and input_gradient.abs().sum()>0
    event('checks','clip_architecture_passed',weights='random tiny model; not an attack result')

if RUN_SELF_CHECKS:
    self_checks()
    clip_architecture_check()
remember_code('checks')


In [ ]:
# Cell 10 — Preflight + immutable run identity. No weight download when RUN_ATTACK=False.
CONFIG_KEYS=['NUM_SAMPLES','SAMPLE_START','RUN_VARIANTS','SEED','BACKBONES','GROUNDING_BACKBONE','CAPTION_MODEL_ID',
    'CAPTION_PROMPT','CAPTION_MAX_NEW_TOKENS','CONCEPT_MAX_NEW_TOKENS','TEXT_RETRIES','MIN_NEGATIVES','MAX_NEGATIVES',
    'MAX_SOURCE_CONCEPTS','MANUAL_TEXT_OVERRIDES','CONCEPT_FAILURE_POLICY','NEGATIVE_FILTER_POLICY','INPUT_RES','STEPS','EPSILON_255','ALPHA_255','CLUSTER_NUMBER',
    'KMEANS_ITERS','SINKHORN_ITERS','OT_REG','FOA_LOCAL_WEIGHT','FUSION_LAMBDA','TEMPERATURE','POSITIVE_OMEGA',
    'LOSS_WEIGHTS','USE_DYNAMIC_ENSEMBLE','HARD_TOP_K','MINE_EVERY','REFRESH_EVERY','GROUNDING_LAST_LAYERS','CROP_AREA',
    'MEMORY_EMA','CREDIT_DECAY','CREDIT_STRENGTH','NEGATIVE_TEXT_SIM_MAX','NEGATIVE_TARGET_MARGIN','USE_AUTOCAST',
    'GRADIENT_SCALE','MOMENTUM','ATTACK_DEVICE','CAPTION_DEVICE']
NOTEBOOK_TEMPLATE_SHA256 = 'kltn-attack-v4.1-h1-concept-fallback-and-attack-preflight'
PAIRS=prepare_pairs()
event('preflight','input_validated',samples=len(PAIRS),caption_images=2*len(PAIRS),variants=RUN_VARIANTS,
      expected_attack_samples=len(PAIRS)*len(RUN_VARIANTS),steps_per_sample=STEPS)
RESOLVED_REVISIONS={}
if RUN_ATTACK:
    assert_device(ATTACK_DEVICE);assert_device(CAPTION_DEVICE)
    if not CODE_SOURCES: raise RuntimeError('Run this as a notebook with execution history available for code fingerprinting.')
    for model_id in [CAPTION_MODEL_ID]+[MODEL_IDS[name] for name in BACKBONES]:
        RESOLVED_REVISIONS[model_id]=HfApi().model_info(model_id,revision=MODEL_REVISION).sha
    config={name:globals()[name] for name in CONFIG_KEYS}
    identity={'notebook_template_sha256':NOTEBOOK_TEMPLATE_SHA256,'config':config,'variants':VARIANTS,'pairs':PAIRS,'model_revisions':RESOLVED_REVISIONS,
              'code_sha256':{name:hashlib.sha256(source.encode()).hexdigest() for name,source in CODE_SOURCES.items()},
              'packages_requested':PACKAGES,'torch':torch.__version__,
              'packages_actual':{name:metadata.version(name) for name in ['transformers','tokenizers','huggingface-hub','torchvision']}}
    fingerprint=json_hash(identity)
    if RESUME_RUN_DIR is not None:
        RUN_DIR=Path(RESUME_RUN_DIR)
        saved=json.loads((RUN_DIR/'run_config.json').read_text(encoding='utf-8'))
        if saved['fingerprint']!=fingerprint: raise ValueError('Resume fingerprint mismatch: config/code/models/data changed. Start a new run.')
    else:
        RUN_DIR=OUTPUT_ROOT/('kltn_v4_h1_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
        RUN_DIR.mkdir(parents=True,exist_ok=False)
        write_json(RUN_DIR/'run_config.json',{'notebook':'kltn-attack-v4.1-h1.ipynb','fingerprint':fingerprint,**identity,
                   'victim_queries_during_attack':0,'research_status':'unvalidated hypothesis; audit generated/fallback concepts and relaxed filters before comparing ASR'})
        (RUN_DIR/'code').mkdir()
        for name,source in CODE_SOURCES.items(): (RUN_DIR/'code'/f'{name}.py').write_text(source,encoding='utf-8')
    EVENT_PATH=RUN_DIR/'events.jsonl'
    event('run','start_or_resume',run_dir=str(RUN_DIR),fingerprint=fingerprint)
else:
    print('Preflight only. Set RUN_ATTACK=True and Run All to generate captions and attack.')


In [ ]:
# Cell 11 — Generate/cache captions and negative edits for every pair, then unload InternVL.
TEXT_RECORDS={}
if RUN_ATTACK:
    text_path=RUN_DIR/'text_records.jsonl'
    for record in read_jsonl(text_path): TEXT_RECORDS[record['sample_id']]=record
    captioner=None;caption_started=time.perf_counter()
    text_status={'status':'running','requested_samples':len(PAIRS),'complete_samples':0,'failed_samples':0}
    try:
        for position,pair in enumerate(PAIRS,1):
            sample_id=pair['sample_id'];cached=TEXT_RECORDS.get(sample_id)
            if cached is not None and cached.get('status')=='ok':
                if cached['pixel_sha256']!=pair['pixel_sha256']: raise ValueError('Cached caption image hash mismatch.')
                text_status['complete_samples']+=1
                event('caption','resume_skip',sample_id=sample_id,samples=f'{position}/{len(PAIRS)}')
                continue
            seed_all(sample_seed(sample_id));record={'sample_id':sample_id,'index':pair['index'],
                     'pixel_sha256':pair['pixel_sha256'],'caption_model':CAPTION_MODEL_ID,
                     'caption_model_revision':RESOLVED_REVISIONS[CAPTION_MODEL_ID]}
            try:
                override=MANUAL_TEXT_OVERRIDES.get(sample_id,{})
                for role in ['source','target']:
                    if hashlib.sha256(Path(pair[role+'_path']).read_bytes()).hexdigest()!=pair['raw_sha256'][role]: raise ValueError('Input image changed.')
                    event('caption','image_start',sample_id=sample_id,role=role,samples=f'{position}/{len(PAIRS)}',
                          image_progress=f"{2*(position-1)+(1 if role=='source' else 2)}/{2*len(PAIRS)}")
                    if role in override: caption=override[role]
                    elif cached is not None and cached.get('pixel_sha256')==pair['pixel_sha256'] and isinstance(cached.get(role),str) and cached[role].strip():
                        caption=cached[role]  # Recover successful image captions from an interrupted/failed text stage.
                        event('caption','reuse_prior_caption',sample_id=sample_id,role=role)
                    else:
                        if captioner is None:
                            event('caption','model_load',model=CAPTION_MODEL_ID)
                            captioner=MPCCaptioner(RESOLVED_REVISIONS[CAPTION_MODEL_ID])
                        pixels=(load_pixels(pair[role+'_path'])[0]*255).round().to(torch.uint8).permute(1,2,0).numpy()
                        caption=captioner.generate(CAPTION_PROMPT,Image.fromarray(pixels))
                    if not isinstance(caption,str) or not caption.strip(): raise ValueError('Empty caption.')
                    record[role]=caption.strip()
                    event('caption','image_complete',sample_id=sample_id,role=role,caption=caption)
                if 'concepts' in override:
                    record['concepts']=parse_concepts(override['concepts'],record['source'],record['target'])
                    record['concepts']['provenance']='manual_override'
                else:
                    if captioner is None: captioner=MPCCaptioner(RESOLVED_REVISIONS[CAPTION_MODEL_ID])
                    record['concepts']=captioner.concepts(record['source'],record['target'],sample_id)
                record['status']='ok';text_status['complete_samples']+=1
            except Exception as error:
                record.update(status='error',error_type=type(error).__name__,error=str(error)[:500])
                text_status['failed_samples']+=1
                event('caption','sample_error',sample_id=sample_id,error_type=type(error).__name__,error=str(error)[:300])
                if isinstance(error,torch.cuda.OutOfMemoryError): raise
            append_jsonl(text_path,record);TEXT_RECORDS[sample_id]=record
            text_status['elapsed_seconds']=time.perf_counter()-caption_started
            write_json(RUN_DIR/'text_status.json',text_status)
            if record['status']=='error' and not CONTINUE_ON_SAMPLE_ERROR: raise RuntimeError('Caption sample failed; artifacts saved.')
        text_status['status']='complete' if text_status['failed_samples']==0 else 'partial'
    except (Exception,KeyboardInterrupt) as error:
        text_status.update(status='interrupted' if isinstance(error,KeyboardInterrupt) else 'error',error_type=type(error).__name__)
        event('caption','stage_stopped',**text_status)
        raise
    finally:
        if captioner is not None: del captioner
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        write_json(RUN_DIR/'text_status.json',text_status)
        event('caption','stage_end',**text_status)


In [ ]:
# Cell 12 — Load only FOA's CLIP models; process one pair across variants.
MODELS={}
if RUN_ATTACK:
    ready_pairs = [p for p in PAIRS if TEXT_RECORDS.get(p['sample_id'], {}).get('status') == 'ok']
    if not ready_pairs:
        raise RuntimeError('No usable text records: no attack steps can execute. Inspect text_status.json/text_records.jsonl, enable CONCEPT_FAILURE_POLICY=deterministic_fallback, or supply manual edits.')
    event('attack','ready_samples',ready=len(ready_pairs),requested=len(PAIRS),backbones=BACKBONES)
    try:
        for name in BACKBONES:
            event('models','clip_load',backbone=name,model_id=MODEL_IDS[name])
            MODELS[name]=ClipBackbone(name,RESOLVED_REVISIONS[MODEL_IDS[name]])
        for position,pair in enumerate(PAIRS,1):
            sample_id=pair['sample_id'];text_record=TEXT_RECORDS.get(sample_id,{})
            pending=[]
            for variant_name in RUN_VARIANTS:
                vdir=RUN_DIR/variant_name;(vdir/'images').mkdir(parents=True,exist_ok=True);(vdir/'samples'/sample_id).mkdir(parents=True,exist_ok=True)
                write_json(vdir/'run_config.json',{'method':variant_name,'attack_experiment':'kltn_v4_h1','parent_run':str(RUN_DIR),
                           'fingerprint':fingerprint,'variant':VARIANTS[variant_name],'config':config})
                status_path=vdir/'samples'/sample_id/'status.json'
                old=json.loads(status_path.read_text()) if status_path.exists() else {}
                completed=old.get('status')=='complete'
                if completed:
                    for role,digest in old['image_file_sha256'].items():
                        image_path=vdir/'images'/f'{sample_id}_{role}.png'
                        if not image_path.is_file() or hashlib.sha256(image_path.read_bytes()).hexdigest()!=digest:
                            completed=False;break
                if completed:
                    event('attack','resume_skip',sample_id=sample_id,variant=variant_name,samples=f'{position}/{len(PAIRS)}')
                else: pending.append(variant_name)
            if not pending: continue
            if text_record.get('status')!='ok':
                for variant_name in pending:
                    write_json(RUN_DIR/variant_name/'samples'/sample_id/'status.json',{'status':'error','error_type':'missing_valid_caption'})
                continue
            try:
                source=load_pixels(pair['source_path']).to(ATTACK_DEVICE);target=load_pixels(pair['target_path']).to(ATTACK_DEVICE)
                if pixel_hash(source)!=pair['pixel_sha256']['source'] or pixel_hash(target)!=pair['pixel_sha256']['target']: raise ValueError('Input pixels changed.')
                anchors,negative_audit=build_anchors(MODELS,source,target,text_record)
            except Exception as error:
                for variant_name in pending:
                    write_json(RUN_DIR/variant_name/'samples'/sample_id/'status.json',{'status':'error','error_type':type(error).__name__,'error':str(error)[:500]})
                event('attack','anchor_error',sample_id=sample_id,error_type=type(error).__name__,error=str(error)[:300])
                if not CONTINUE_ON_SAMPLE_ERROR or isinstance(error,torch.cuda.OutOfMemoryError): raise
                continue
            for variant_name in pending:
                seed_all(sample_seed(sample_id))
                vdir=RUN_DIR/variant_name;sample_dir=vdir/'samples'/sample_id
                write_json(sample_dir/'negative_candidates.json',{'concepts':text_record['concepts'],**negative_audit})
                spans=MODELS[GROUNDING_BACKBONE].token_spans(text_record['source'],text_record['concepts']['source_concepts'])
                write_json(sample_dir/'concept_token_spans.json',spans)
                write_json(sample_dir/'captions.json',text_record)
                status={'status':'running','sample_id':sample_id,'completed_steps':0,'total_steps':STEPS}
                write_json(sample_dir/'status.json',status)
                event('attack','sample_start',sample_id=sample_id,variant=variant_name,samples=f'{position}/{len(PAIRS)}')
                def checkpoint_status(step,row):
                    status.update(completed_steps=step,objective=row['objective'],box=row['box'])
                    write_json(sample_dir/'status.json',status)
                try:
                    adversarial,stats=run_sample(MODELS,anchors,source,target,text_record,variant_name,sample_dir,checkpoint_status)
                    for role,tensor in [('source',source),('target',target),('adversarial',adversarial)]:
                        if role=='adversarial': save_adversarial_pixels(vdir/'images'/f'{sample_id}_{role}.png',tensor,source)
                        else: save_pixels(vdir/'images'/f'{sample_id}_{role}.png',tensor)
                    src=np.asarray(Image.open(vdir/'images'/f'{sample_id}_source.png')).astype(np.int16)
                    adv=np.asarray(Image.open(vdir/'images'/f'{sample_id}_adversarial.png')).astype(np.int16)
                    linf=int(np.abs(adv-src).max())
                    if linf>EPSILON_255+1e-6: raise AssertionError('Saved PNG violates quantized L-infinity bound.')
                    record={'index':pair['index'],'sample_id':sample_id,'worker_rank':0,'original_path':pair['source_path'],
                            'target_path':pair['target_path'],'linf_255':linf,'method':variant_name,
                            'concept_provenance':text_record['concepts'].get('provenance','unknown'),
                            'negative_filter_relaxed':negative_audit.get('negative_filter_relaxed',False),'attack_experiment':'kltn_v4_h1',
                            **{role:f'images/{sample_id}_{role}.png' for role in ['source','target','adversarial']}}
                    # Per-sample record is authoritative; Cell 13 rebuilds manifests without duplicates.
                    write_json(sample_dir/'record.json',record)
                    digests={role:hashlib.sha256((vdir/record[role]).read_bytes()).hexdigest() for role in ['source','target','adversarial']}
                    status.update(status='complete',**stats,image_file_sha256=digests,linf_255=linf)
                    event('attack','sample_complete',sample_id=sample_id,variant=variant_name,linf_255=linf,seconds=round(stats['seconds'],2))
                except (Exception,KeyboardInterrupt) as error:
                    status.update(status='interrupted' if isinstance(error,KeyboardInterrupt) else 'error',error_type=type(error).__name__,error=str(error)[:500])
                    event('attack','sample_stopped',variant=variant_name,**status)
                    if isinstance(error,(KeyboardInterrupt,torch.cuda.OutOfMemoryError)) or not CONTINUE_ON_SAMPLE_ERROR: raise
                finally:
                    write_json(sample_dir/'status.json',status)
            del anchors,source,target
    finally:
        MODELS.clear();gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()


In [ ]:
# Cell 13 — Rebuild manifests and export, also runnable after an interrupted attack cell.
if RUN_DIR is not None:
    overall={}
    for variant_name in RUN_VARIANTS:
        vdir=RUN_DIR/variant_name
        if not vdir.exists(): continue
        records=[];errors=[]
        for pair in PAIRS:
            directory=vdir/'samples'/pair['sample_id'];path=directory/'status.json'
            state=json.loads(path.read_text()) if path.exists() else {'status':'pending'}
            if state['status']=='complete': records.append(json.loads((directory/'record.json').read_text()))
            else: errors.append({'sample_id':pair['sample_id'],**state})
        (vdir/'manifest.jsonl').write_text(''.join(json.dumps(r,ensure_ascii=False)+'\n' for r in records),encoding='utf-8')
        summary={'status':'complete' if len(records)==len(PAIRS) else 'partial','samples':len(records),'requested_samples':len(PAIRS),
                 'failed_or_pending':len(errors),'coverage':len(records)/len(PAIRS),'method':variant_name,
                 'victim_queries_during_attack':0,'ASR':None,'ASR_note':'Run held-out victim captioning and GPT judge separately.',
                 'epsilon_255':EPSILON_255,'steps':STEPS}
        write_json(vdir/'summary.json',summary);write_json(vdir/'status.json',summary);write_json(vdir/'failures.json',errors)
        overall[variant_name]=summary
        print(variant_name,summary)
    write_json(RUN_DIR/'summary.json',overall)
    if all(s['samples']==0 for s in overall.values()):
        event('attack','zero_completed_samples',reason='All samples skipped/failed; inspect failures.json and text_records.jsonl')
    write_json(RUN_DIR/'status.json',{'status':'complete' if len(overall)==len(RUN_VARIANTS) and all(s['status']=='complete' for s in overall.values()) else 'partial',
                                    'requested_samples':len(PAIRS),'requested_variants':RUN_VARIANTS,'variants':overall})
    if EXPORT_ZIP:
        archive=shutil.make_archive(str(RUN_DIR),'zip',root_dir=RUN_DIR)
        print('ZIP:',archive)
        try:
            from IPython.display import FileLink, display
            display(FileLink(archive))
        except ImportError: pass
else:
    print('No run created. RUN_ATTACK=False: input validation only.')


## 3. Inspect và đánh giá giả thuyết

Cell sau hiển thị source/target/adv, relevance ở các refresh và trajectory loss. Review negative captions trong `samples/<ID>/negative_candidates.json`; không diễn giải heatmap là segmentation ground truth. Fallback uniform được ghi rõ và phải báo tỷ lệ khi phân tích.

Để tính **transfer ASR**, dùng folder từng variant làm `INPUT_DIR` của notebook text evaluation, `NUM_SAMPLES=None`, cùng victim/caption prompt/judge/threshold cho mọi variant và baseline. Với paper-style targeted rule chọn **score > 0.5**; nếu giữ repository v1 `>=0.3`, ghi rõ và không trộn hai protocol. Source/target/adv phải được caption bởi **cùng victim model** lúc evaluation; InternVL captions ở đây chỉ dùng crafting.

Đánh giá cùng source-target pairs, epsilon, steps, seeds; báo ASR kèm số mẫu hợp lệ/lỗi, AvgSim, L∞/L2, thời gian/VRAM, số forward/backward và grounding refresh. Dynamic tốn thêm compute: ngoài equal-step ablation cần **equal-wall-time hoặc equal-forward-budget**. Chạy nhiều seeds; không tune hyperparameters bằng test victim rồi gọi đó là zero-query. So với FOA notebook nguyên bản, MPC-CLIP control và fixed/fresh/history, hard/random/no-negative, thêm `h1_no_hard_loss` để giữ nguyên negative-conditioned grounding. `h1_no_fusion` chỉ đo đóng góp fused-anchor loss.

Chỉ kết luận hỗ trợ H1 khi hard-bank vượt no-bank/random-bank trên held-out victims; H2 khi grounded vượt random với area bằng nhau; H3 khi refresh/history vượt fixed/fresh sau kiểm soát compute; H4 khi fused anchors vượt image-only anchors (`h1_image_anchors`) với các loss/grounding khác giữ nguyên. Notebook **chưa chạy pretrained attack hoặc đánh giá black-box** trong quá trình authoring.

Gợi ý chạy ablation (Cell 1):
```python
RUN_VARIANTS = ['h1_full', 'h1_no_hard_loss', 'h1_random_neg',
                'h1_random', 'h1_fixed', 'h1_dynamic', 'h1_image_anchors']
```
Chạy tất cả variants tốn thời gian tương ứng; caption/negative bank được dùng chung. Khi so với baseline trước đây, truyền `PAIR_MANIFEST` từ baseline để giữ chính xác `index`/`sample_id`.

**Dữ liệu đã đọc trong repo:** `src/FOA-Attack/surrogates/FeatureExtractors/{ClipB16,ClipB32,ClipLaion,Base}.py`; `src/MPCAttack/generate_adversarial_examples_MPCAttack.py`, `utils.py`, `surrogates/FeatureExtractors/InternVL3_1B.py`; notebook attack v3.2 và evaluator v2. Source/target trong manifest cũ là `bigscale/nips17/<stem>.png` và `target_images/1/<stem>.jpg`; code pairing kiểm tra stems trước khi chạy. Trang Kaggle không đọc được bằng công cụ web trong lượt này, chưa tải toàn bộ dataset để xác minh inventory.

`compute_counts` trong mỗi `samples/<ID>/status.json` đếm vision forward, attention forward, attack backward và attribution backward cho vòng tối ưu; không bao gồm giai đoạn sinh caption và build anchors. `union_support_fraction` cho biết vùng đã từng được cập nhật qua các block; `peak_cuda_allocated_gib` bao gồm model resident. `concept_maps_*.npy` có thứ tự source concepts trước, active negative captions sau; IDs negatives là vị trí trong bank sau filter (`candidate_indices` ánh xạ về bank gốc).


**Kiểm tra khi tạo notebook:** schema/cú pháp và integration trên ảnh tổng hợp với hai HF CLIP nhỏ, random weights, Transformers 4.57.1: cả 11 variants chạy 3 bước, gradient attention/input, refresh/fixed/random regions, giới hạn PNG, resume mẫu hoàn tất, ZIP/manifest và các nhánh caption/optimizer lỗi. Đây là kiểm tra tính đúng của đường chạy; không đánh giá pretrained transfer. `RUN_SELF_CHECKS=True` chạy lại kiểm tra API/gradient bằng tiny CLIP không tải weights trong Cell 9.


In [ ]:
# Cell 14 — Optional artifact-only preview; no models or APIs.
PREVIEW_VARIANT = RUN_VARIANTS[0]
PREVIEW_SAMPLE_ID = '00000'
if RUN_DIR is not None and (RUN_DIR/PREVIEW_VARIANT/'samples'/PREVIEW_SAMPLE_ID/'record.json').is_file():
    import matplotlib.pyplot as plt
    from matplotlib.patches import Rectangle
    vdir=RUN_DIR/PREVIEW_VARIANT;sample_dir=vdir/'samples'/PREVIEW_SAMPLE_ID
    record=json.loads((sample_dir/'record.json').read_text())
    fig,axes=plt.subplots(1,3,figsize=(12,4))
    for axis,role in zip(axes,['source','target','adversarial']):
        axis.imshow(Image.open(vdir/record[role]));axis.set_title(role);axis.axis('off')
    plt.show()
    trace=read_jsonl(sample_dir/'steps.jsonl')
    plt.figure(figsize=(8,3));plt.plot([r['step'] for r in trace],[r['objective'] for r in trace]);plt.xlabel('step');plt.ylabel('surrogate loss');plt.show()
    paths=sorted((sample_dir/'maps').glob('relevance_*.npy'))
    if paths:
        chosen=sorted(set([0,len(paths)//2,len(paths)-1]));fig,axes=plt.subplots(1,len(chosen),figsize=(5*len(chosen),4),squeeze=False)
        for axis,i in zip(axes[0],chosen):
            path=paths[i];state=json.loads(path.with_suffix('.json').read_text());step=state['computed_after_steps']
            axis.imshow(Image.open(sample_dir/'maps'/f'image_{step:05d}.png'))
            axis.imshow(np.load(path).squeeze(),cmap='magma',alpha=0.5)
            top,left,height,width=state['box'];axis.add_patch(Rectangle((left,top),width,height,fill=False,edgecolor='cyan'))
            axis.set_title(f're-ground after {step} steps');axis.axis('off')
        plt.show()
    print(json.loads((sample_dir/'captions.json').read_text()))
else:
    print('Set RUN_DIR/PREVIEW_VARIANT/PREVIEW_SAMPLE_ID to an existing completed sample for preview.')
